# PolitCheck – Datenbank Explorer

Dieses Notebook liest Daten aus beiden SQLite-Datenbanken via SQL aus.

| Datenbank | Pfad | Inhalt |
|-----------|------|--------|
| Rohdaten  | `data/plenarprotokolle.db` | API-Metadaten aller Plenarprotokolle |
| Analyse   | `data/politcheck.db`       | LLM-extrahierte Aussagen + Politiker-Stats |

In [1]:
pip install pandas sqlite3 json pathlib

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for sqlite3


In [2]:
import sqlite3
import pandas as pd
import json
from pathlib import Path

# Pfade relativ zum Notebook (eda/ -> data/)
DB_RAW      = Path('../../data/plenarprotokolle.db')
DB_ANALYSE  = Path('../../data/politcheck.db')

def query(db_path: Path, sql: str, params=()) -> pd.DataFrame:
    """Führt eine SQL-Abfrage aus und gibt ein DataFrame zurück."""
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(sql, conn, params=params)

print('Rohdaten-DB existiert:', DB_RAW.exists())
print('Analyse-DB existiert: ', DB_ANALYSE.exists())

Rohdaten-DB existiert: True
Analyse-DB existiert:  True


---
## 1. Rohdaten-DB – `plenarprotokolle.db`

In [3]:
# Übersicht: Gesamtanzahl, Zeitraum
query(DB_RAW, """
    SELECT
        COUNT(*)       AS gesamt,
        MIN(datum)     AS aeltestes_datum,
        MAX(datum)     AS neuestes_datum
    FROM protokolle
""")

,gesamt,aeltestes_datum,neuestes_datum
0,5779,1949-09-07,2026-05-22


In [4]:
# Protokolle pro Jahr und Monat
query(DB_RAW, """
    SELECT
        strftime('%Y', datum)       AS jahr,
        strftime('%m', datum)       AS monat,
        COUNT(*)                    AS anzahl
    FROM protokolle
    WHERE datum IS NOT NULL
    GROUP BY jahr, monat
    ORDER BY jahr DESC, monat DESC
""")

,jahr,monat,anzahl
0,2026,05,7
1,2026,04,7
2,2026,03,11
3,2026,02,3
4,2026,01,7
...,...,...,...
843,1950,01,9
844,1949,12,9
845,1949,11,8
846,1949,10,3


In [11]:
# Protokolle pro Wahlperiode
query(DB_RAW, """
    SELECT
        wahlperiode,
      min(strftime('%Y', datum)) as start_date,
      max(strftime('%Y', datum)) as end_date,
        COUNT(*) AS anzahl
    FROM protokolle
    GROUP BY wahlperiode
    ORDER BY wahlperiode DESC
""")

,wahlperiode,start_date,end_date,anzahl
0,21.0,2025,2026,94
1,20.0,2021,2025,258
2,19.0,2017,2021,291
3,18.0,2013,2017,294
4,17.0,2009,2013,317
5,16.0,2005,2009,281
6,15.0,2002,2005,223
7,14.0,1998,2002,307
8,13.0,1994,1998,307
9,12.0,1990,1994,307


In [6]:
# Neueste 20 Protokolle
query(DB_RAW, """
    SELECT *
    FROM protokolle
    ORDER BY datum DESC
    LIMIT 20
""")

,id,titel,datum,wahlperiode,sitzungsnr,pdf_url,abstract,raw_json,gespeichert_am
0,5796,Protokoll der 81. Sitzung des 21. Deutschen Bu...,2026-05-22,21,21/81,https://dserver.bundestag.de/btp/21/21081.pdf,None,"{""id"": ""5796"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968
1,5795,Protokoll der 80. Sitzung des 21. Deutschen Bu...,2026-05-21,21,21/80,https://dserver.bundestag.de/btp/21/21080.pdf,None,"{""id"": ""5795"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968
2,5794,Protokoll der 79. Sitzung des 21. Deutschen Bu...,2026-05-20,21,21/79,https://dserver.bundestag.de/btp/21/21079.pdf,None,"{""id"": ""5794"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968
3,5793,Protokoll der 1065. Sitzung des Bundesrates,2026-05-08,21,1065,https://dserver.bundestag.de/brp/1065.pdf,None,"{""id"": ""5793"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968
4,5792,Protokoll der 78. Sitzung des 21. Deutschen Bu...,2026-05-08,21,21/78,https://dserver.bundestag.de/btp/21/21078.pdf,None,"{""id"": ""5792"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968
5,5791,Protokoll der 77. Sitzung des 21. Deutschen Bu...,2026-05-07,21,21/77,https://dserver.bundestag.de/btp/21/21077.pdf,None,"{""id"": ""5791"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968
6,5790,Protokoll der 76. Sitzung des 21. Deutschen Bu...,2026-05-06,21,21/76,https://dserver.bundestag.de/btp/21/21076.pdf,None,"{""id"": ""5790"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968
7,5789,Protokoll der 1064. Sitzung des Bundesrates,2026-04-24,21,1064,https://dserver.bundestag.de/brp/1064.pdf,None,"{""id"": ""5789"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968
8,5788,Protokoll der 75. Sitzung des 21. Deutschen Bu...,2026-04-24,21,21/75,https://dserver.bundestag.de/btp/21/21075.pdf,None,"{""id"": ""5788"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968
9,5787,Protokoll der 74. Sitzung des 21. Deutschen Bu...,2026-04-23,21,21/74,https://dserver.bundestag.de/btp/21/21074.pdf,None,"{""id"": ""5787"", ""dokumentart"": ""Plenarprotokoll...",2026-05-27T20:20:51.333968


In [7]:
# Protokolle eines bestimmten Zeitraums filtern
query(DB_RAW, """
    SELECT
        id,
        datum,
        titel,
        abstract
    FROM protokolle
    WHERE datum BETWEEN '2025-01-01' AND '2025-12-31'
    ORDER BY datum DESC
""")

,id,datum,titel,abstract
0,5761,2025-12-19,Protokoll der 1060. Sitzung des Bundesrates,None
1,5760,2025-12-19,Protokoll der 51. Sitzung des 21. Deutschen Bu...,None
2,5759,2025-12-18,Protokoll der 50. Sitzung des 21. Deutschen Bu...,None
3,5758,2025-12-17,Protokoll der 49. Sitzung des 21. Deutschen Bu...,None
4,5757,2025-12-05,Protokoll der 48. Sitzung des 21. Deutschen Bu...,None
...,...,...,...,...
62,5699,2025-02-14,Protokoll der 1051. Sitzung des Bundesrates,None
63,5698,2025-02-11,Protokoll der 212. Sitzung des 20. Deutschen B...,None
64,5697,2025-01-31,Protokoll der 211. Sitzung des 20. Deutschen B...,None
65,5696,2025-01-30,Protokoll der 210. Sitzung des 20. Deutschen B...,None


In [8]:
# Ein einzelnes Protokoll vollständig anzeigen (inkl. raw_json)
df = query(DB_RAW, """
    SELECT * FROM protokolle
    ORDER BY datum DESC
    LIMIT 1
""")

# raw_json schön formatieren
row = df.iloc[0]
print('Titel:         ', row['titel'])
print('Datum:         ', row['datum'])
print('PDF-URL:       ', row['pdf_url'])
print('Abstract:      ', str(row['abstract'])[:200])
print()
print('--- raw_json (Auszug) ---')
raw = json.loads(row['raw_json'])
print(json.dumps(raw, indent=2, ensure_ascii=False)[:1000])

Titel:          Protokoll der 81. Sitzung des 21. Deutschen Bundestages
Datum:          2026-05-22
PDF-URL:        https://dserver.bundestag.de/btp/21/21081.pdf
Abstract:       None

--- raw_json (Auszug) ---
{
  "id": "5796",
  "dokumentart": "Plenarprotokoll",
  "typ": "Dokument",
  "vorgangsbezug_anzahl": 15,
  "dokumentnummer": "21/81",
  "wahlperiode": 21,
  "herausgeber": "BT",
  "pdf_hash": "f9536b0264e9f4d080d336ac305589c8",
  "xml_hash": "8fe1b5d3f9da2477cacfe589809dda6f",
  "aktualisiert": "2026-05-26T11:04:03+02:00",
  "vorgangsbezug": [
    {
      "id": "321621",
      "titel": "Bericht der Bundesregierung zum Stand der Bemühungen um Rüstungskontrolle, Abrüstung und Nichtverbreitung sowie über die Entwicklung der Streitkräftepotenziale für das Jahr 2024 (Jahresabrüstungsbericht 2024)",
      "vorgangstyp": "Bericht, Gutachten, Programm"
    },
    {
      "id": "325510",
      "titel": "Nationaler Aktionsplan \"Neue Chancen für Kinder in Deutschland\"",
      "vorgangstyp"

In [9]:
# Protokolle nach Import-Datum (gespeichert_am) – wann wurden welche Daten importiert?
query(DB_RAW, """
    SELECT
        DATE(gespeichert_am) AS import_datum,
        COUNT(*)             AS anzahl
    FROM protokolle
    GROUP BY DATE(gespeichert_am)
    ORDER BY import_datum DESC
    LIMIT 10
""")

,import_datum,anzahl
0,2026-05-27,5779


---
## 2. Analyse-DB – `politcheck.db`

In [10]:
# Übersicht Analyse-DB
query(DB_ANALYSE, """
    SELECT
        (SELECT COUNT(*) FROM aussagen)             AS aussagen_gesamt,
        (SELECT COUNT(*) FROM verarbeitete_quellen) AS verarbeitete_protokolle
""")

DatabaseError: Execution failed on sql '
    SELECT
        (SELECT COUNT(*) FROM aussagen)             AS aussagen_gesamt,
        (SELECT COUNT(*) FROM verarbeitete_quellen) AS verarbeitete_protokolle
': no such table: aussagen

In [ ]:
# Top 20 Aussagen nach Polarisierungsgrad
query(DB_ANALYSE, """
    SELECT
        politiker,
        partei,
        datum,
        polarisierungsgrad,
        thema,
        aussage
    FROM aussagen
    ORDER BY polarisierungsgrad DESC
    LIMIT 20
""")

In [ ]:
# Aussagen nach Thema gruppiert
query(DB_ANALYSE, """
    SELECT
        thema,
        COUNT(*)                              AS anzahl,
        ROUND(AVG(polarisierungsgrad), 1)     AS avg_polarisierung,
        MAX(polarisierungsgrad)               AS max_polarisierung
    FROM aussagen
    GROUP BY thema
    ORDER BY anzahl DESC
""")

In [ ]:
# Politiker-Ranking nach durchschnittlichem Polarisierungsgrad
query(DB_ANALYSE, """
    SELECT
        politiker,
        partei,
        COUNT(*)                              AS anzahl_aussagen,
        ROUND(AVG(polarisierungsgrad), 1)     AS avg_polarisierung,
        MAX(polarisierungsgrad)               AS max_polarisierung
    FROM aussagen
    GROUP BY politiker, partei
    HAVING anzahl_aussagen >= 2
    ORDER BY avg_polarisierung DESC
    LIMIT 20
""")

In [ ]:
# Aussagen nach Partei filtern (z.B. AfD)
partei_filter = 'AfD'  # <-- hier anpassen

query(DB_ANALYSE, """
    SELECT
        politiker,
        datum,
        polarisierungsgrad,
        thema,
        aussage
    FROM aussagen
    WHERE partei = ?
    ORDER BY polarisierungsgrad DESC
    LIMIT 20
""", params=(partei_filter,))

In [ ]:
# Aussagen nach Thema filtern
thema_filter = 'Migration'  # <-- hier anpassen

query(DB_ANALYSE, """
    SELECT
        politiker,
        partei,
        datum,
        polarisierungsgrad,
        aussage,
        polarisierungsbegruendung
    FROM aussagen
    WHERE thema = ?
    ORDER BY polarisierungsgrad DESC
""", params=(thema_filter,))

In [ ]:
# Verteilung der Polarisierungsgrade (Histogramm-Daten)
query(DB_ANALYSE, """
    SELECT
        polarisierungsgrad,
        COUNT(*) AS anzahl
    FROM aussagen
    GROUP BY polarisierungsgrad
    ORDER BY polarisierungsgrad DESC
""")

In [ ]:
# Verarbeitete Quellen – welche Protokoll-IDs wurden bereits durch Workflow 2 verarbeitet?
query(DB_ANALYSE, """
    SELECT
        quelle_id,
        quelle_typ,
        verarbeitet_am
    FROM verarbeitete_quellen
    ORDER BY verarbeitet_am DESC
    LIMIT 20
""")

---
## 3. Cross-DB: Welche Protokolle wurden noch NICHT verarbeitet?

In [ ]:
# Da zwei separate DBs: Python-seitig joinen
with sqlite3.connect(DB_RAW) as conn_raw:
    alle_ids = set(
        r[0] for r in conn_raw.execute('SELECT id FROM protokolle').fetchall()
    )

with sqlite3.connect(DB_ANALYSE) as conn_analyse:
    verarbeitete_ids = set(
        r[0] for r in conn_analyse.execute('SELECT quelle_id FROM verarbeitete_quellen').fetchall()
    )

nicht_verarbeitet = alle_ids - verarbeitete_ids

print(f'Protokolle in Rohdaten-DB total:       {len(alle_ids)}')
print(f'Davon bereits durch Workflow 2 extrahiert: {len(verarbeitete_ids)}')
print(f'Noch ausstehend (Workflow 2):              {len(nicht_verarbeitet)}')

In [ ]:
# Details zu noch nicht verarbeiteten Protokollen (neueste zuerst)
if nicht_verarbeitet:
    placeholders = ','.join('?' * len(nicht_verarbeitet))
    with sqlite3.connect(DB_RAW) as conn:
        df = pd.read_sql_query(
            f'SELECT id, datum, dokumentnummer, titel FROM protokolle WHERE id IN ({placeholders}) ORDER BY datum DESC LIMIT 20',
            conn,
            params=list(nicht_verarbeitet)
        )
    df
else:
    print('Alle Protokolle wurden bereits verarbeitet.')